# 07 Model Evaluation

In [ ]:
!rm -rf /content/unet-ensemble

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
sys.path.append('/content/unet-ensemble')

In [4]:
!pip install safetensors huggingface_hub segmentation-models-pytorch scikit-image scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.1 MB/s eta 0:00:00


In [5]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import login
from torch.utils.data import DataLoader

from src.dataset.dataset     import Dataset as LazyDS
from src.dataset.dataloader  import DataLoader as DSLoader
from src.utils.evaluate import Evaluate

In [6]:
token = ''
login(token=token)

## 07-1 Configuration

In [ ]:

IMG_SIZE     = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE   = 8
NUM_WORKERS  = 2

HF_USERNAME = 'hf-username'
MODEL_REPO  = f'{HF_USERNAME}/mben-unetpp'
SUBFOLDER   = 'all_features'

# Choose ONE of the four supported combinations (must match the saved model):
#   ('prnu', 'illumination', 'frequency')  — all three features
#   ('prnu', 'frequency')                  — PRNU + Frequency
#   ('prnu', 'illumination')               — PRNU + Illumination
#   ('frequency', 'illumination')          — Frequency + Illumination
FEATURES = ('prnu', 'illumination', 'frequency')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 07-2 Dataset Preparation

In [9]:
MASK_FOLDER         = 'GTA - Masks'
PRNU_FOLDER         = 'Feature - PRNU'
ILLUMINATION_FOLDER = 'Feature - Illumination'
FREQUENCY_FOLDER    = 'Feature - Frequency'
CATEGORIES          = ['Synthetic', 'Authentic']
TEMPLATES           = [
    'template-a', 'template-albania', 'template-b',
    'template-c', 'template-chile',   'template-deutschland',
    'template-spain', 'template-usa',
]

In [ ]:
ds_loader = DSLoader(
    mask_folder         = MASK_FOLDER,
    prnu_folder         = PRNU_FOLDER,
    illumination_folder = ILLUMINATION_FOLDER,
    frequency_folder    = FREQUENCY_FOLDER,
    categories          = CATEGORIES,
    templates           = TEMPLATES,
    features            = FEATURES,
)

test_samples = ds_loader.load_images('Test', DATASET_ROOT)
test_ds      = LazyDS(test_samples, IMG_SIZE, augment=False, features=FEATURES)

test_loader = DataLoader(
    test_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

print(f'Features : {sorted(FEATURES)}')
print(f'Test batches: {len(test_loader)}')

## 07-3 Evaluation

In [ ]:
evaluator = Evaluate(device=device, features=FEATURES)

model   = evaluator.load_unetpp_from_hub(MODEL_REPO, SUBFOLDER)  # ← swap loader method if needed
metrics = evaluator.run(test_loader, model)

Evaluate.print_metrics(metrics, label=f'{MODEL_REPO} | {sorted(FEATURES)}')

In [ ]:
results_df = pd.DataFrame([{'Features': '+'.join(sorted(FEATURES)), **metrics}])
pd.set_option('display.float_format', '{:.4f}'.format)
display(results_df)

In [ ]:
import os

csv_path = '/content/drive/Shareddrives/U-Net Ensemble/evaluation_results.csv'

row = pd.DataFrame([{
    'Model'    : MODEL_REPO,
    'Subfolder': SUBFOLDER,
    'Features' : '+'.join(sorted(FEATURES)),
    **metrics
}])

if os.path.exists(csv_path):
    row.to_csv(csv_path, mode='a', header=False, index=False)
else:
    row.to_csv(csv_path, mode='w', header=True, index=False)

print(f'Results appended to {csv_path}')

## 07-5 Metrics Visualization

In [ ]:
METRICS_TO_PLOT = ['IoU', 'Dice', 'Pixel_Accuracy', 'BF_Score']
vals = [metrics[m] for m in METRICS_TO_PLOT]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(METRICS_TO_PLOT, vals, width=0.5)
ax.bar_label(bars, fmt='{:.4f}', padding=3, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title(f'Evaluation — {MODEL_REPO} | {sorted(FEATURES)}')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150)
plt.show()